# sys1 v2: calibrated-decision training

`train_lora.ipynb` (v1) trains with plain next-token cross-entropy over
the *whole vocabulary* — it makes the model more likely to say the right
letter, but nothing in that loss cares whether the resulting probability
is calibrated. This notebook trains directly on the exact distribution
`src/sys1/engine.py` reads at inference time: softmax over just the
candidate-letter logits. That distribution is fully differentiable in a
single forward pass, so plain gradient descent on it *is* the calibration
objective — no reward model or policy-gradient RL loop needed for a
single-step decision like this.

Steps:
1. Upload `data/prepared_calibrated/train.jsonl` and `val.jsonl` (from
   `training/prepare_calibrated_dataset.py`) to this Colab session.
2. Runtime > Change runtime type > GPU, then run all cells.
3. Download `sys1-calibrated-out/` back to this machine, same as v1.

In [ ]:
!pip install -q -U transformers accelerate peft datasets torchao

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_FILE = "/content/train.jsonl"
VAL_FILE = "/content/val.jsonl"
OUT_DIR = "sys1-calibrated-out"
NUM_EPOCHS = 3
BATCH_SIZE = 4
LEARNING_RATE = 2e-4

In [ ]:
import json

def load_rows(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = load_rows(TRAIN_FILE)
val_rows = load_rows(VAL_FILE)
len(train_rows), len(val_rows)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "left"  # so the last real token is always at position -1
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16).to("cuda")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
def candidate_token_id(label):
    for variant in (label, " " + label):
        ids = tokenizer.encode(variant, add_special_tokens=False)
        if len(ids) >= 1:
            return ids[0]
    return None

# All examples use letters A, B, C, ... up to the widest option set seen.
MAX_LETTERS = max(len(r["letters"]) for r in train_rows + val_rows)
ALL_LETTERS = [chr(ord("A") + i) for i in range(MAX_LETTERS)]
LETTER_TOKEN_IDS = [candidate_token_id(l) for l in ALL_LETTERS]
assert all(t is not None for t in LETTER_TOKEN_IDS)
LETTER_TOKEN_IDS = torch.tensor(LETTER_TOKEN_IDS, device="cuda")
MAX_LETTERS

In [ ]:
import torch.nn.functional as F
from torch.utils.data import DataLoader

def collate(batch):
    prompts = [r["prompt"] for r in batch]
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    n_candidates = torch.tensor([len(r["letters"]) for r in batch], device="cuda")
    answer_index = torch.tensor([r["answer_index"] for r in batch], device="cuda")
    return enc, n_candidates, answer_index

def calibrated_loss(logits_last, n_candidates, answer_index):
    """logits_last: (batch, vocab) logits at the next-token position.
    Slice out this batch's candidate-letter logits, mask entries past
    each example's actual option count with -inf so they don't affect
    the softmax denominator, then cross-entropy against the true index —
    exactly the distribution src/sys1/engine.py reads at inference."""
    candidate_logits = logits_last[:, LETTER_TOKEN_IDS[: n_candidates.max()]]
    mask = torch.arange(candidate_logits.size(1), device="cuda")[None, :] >= n_candidates[:, None]
    candidate_logits = candidate_logits.masked_fill(mask, float("-inf"))
    return F.cross_entropy(candidate_logits, answer_index), candidate_logits

train_loader = DataLoader(train_rows, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
val_loader = DataLoader(val_rows, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

In [ ]:
@torch.no_grad()
def evaluate():
    model.eval()
    n_correct = n_total = 0
    brier_total = 0.0
    for enc, n_candidates, answer_index in val_loader:
        logits_last = model(**enc).logits[:, -1, :]
        _, candidate_logits = calibrated_loss(logits_last, n_candidates, answer_index)
        probs = torch.softmax(candidate_logits, dim=-1)
        preds = probs.argmax(dim=-1)
        n_correct += (preds == answer_index).sum().item()
        n_total += len(answer_index)
        one_hot = F.one_hot(answer_index, probs.size(1)).float()
        # zero out the padded candidate slots before scoring Brier
        valid = torch.arange(probs.size(1), device="cuda")[None, :] < n_candidates[:, None]
        brier_total += (((probs - one_hot) ** 2) * valid).sum().item()
    model.train()
    return n_correct / n_total, brier_total / n_total

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

acc, brier = evaluate()
print(f"before training: accuracy={acc:.2%} brier={brier:.4f}")

for epoch in range(NUM_EPOCHS):
    for step, (enc, n_candidates, answer_index) in enumerate(train_loader):
        logits_last = model(**enc).logits[:, -1, :]
        loss, _ = calibrated_loss(logits_last, n_candidates, answer_index)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if step % 20 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f}")
    acc, brier = evaluate()
    print(f"epoch {epoch} done: val accuracy={acc:.2%} brier={brier:.4f}")

In [ ]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
# !zip -r sys1-calibrated-out.zip sys1-calibrated-out
# then download the zip and extract into models/ locally, same as v1.